In [1]:
import geopandas as gpd
from shapely.geometry import Polygon, multipolygon
from shapely.ops import unary_union, snap
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import mapping, shape
from shapely.errors import TopologicalError
from shapely.validation import make_valid
from shapely import wkt
import json
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
gdf = gpd.read_file(r"C:\Users\I33238\Desktop\Italy\estonia-251106-free\Result_LU\R1\P1.shp")
                    #"C:\Users\I33238\Desktop\Italy\S_NI_ND_1.gpkg")
    #r"C:\Users\I33238\Desktop\Austria\Eubucco_AUT\Test\T1.shp")#, layer="v0_1aut" )
gdf['geometry'] = gdf['geometry'].apply(lambda geom: geom if geom.is_valid else geom.buffer(0))
gdf = gdf.drop_duplicates(subset='geometry') #Remove duplicates 
spatial_index = gdf.sindex #Index to speedup the process

if 'fid' in gdf.columns:
    gdf = gdf.rename(columns={'fid': 'fid_orig'})


def find_overlaps(geometry):
    possible_matches_index = list(spatial_index.intersection(geometry.bounds))
    possible_matches = gdf.iloc[possible_matches_index]
    return possible_matches[possible_matches.geometry.overlaps(geometry)]

overlap_indices = gdf.geometry.apply(lambda x: not find_overlaps(x).empty)
overlap_gdf = gdf[overlap_indices]  # Overlapping polygons
non_overlap_gdf = gdf[~overlap_indices]  # Non-overlapping polygons


intersected_geometries = []
processed_geometries = []

for index, row in overlap_gdf.iterrows():
    overlaps = find_overlaps(row.geometry)
    for _, overlap_row in overlaps.iterrows():
        intersection = row.geometry.intersection(overlap_row.geometry)  # Intersection
        buffered_intersection = intersection.buffer(0.01)  # Buffer the intersection
        difference = row.geometry.difference(buffered_intersection)  # Difference from original
        intersected_geometries.append(buffered_intersection)

        #new_row=row.copy()
        #new_row.geometry= buffered_intersection
        #intersected_geometries.append(new_row)
    #new_row=row.copy()
    #new_row.geometry= difference
    processed_geometries.append(difference)


intersected_gdf = gpd.GeoDataFrame(geometry=intersected_geometries, crs=gdf.crs)
processed_gdf = gpd.GeoDataFrame(geometry=processed_geometries, crs=gdf.crs)


overlap_gdf = overlap_gdf[overlap_gdf.is_valid]
non_overlap_gdf = non_overlap_gdf[non_overlap_gdf.is_valid] #
intersected_gdf = intersected_gdf[intersected_gdf.is_valid]
processed_gdf = processed_gdf[processed_gdf.is_valid]
#for df in [overlap_gdf, non_overlap_gdf, intersected_gdf, processed_gdf]:
#    df = df[df.is_valid]
#gdf["geometry"] = gdf["geometry"].make_valid() #check it for shapely functions

#merged_gdf = gpd.GeoDataFrame(pd.concat([non_overlap_gdf, intersected_gdf], ignore_index=True), crs=gdf.crs)
#merged_gdf = merged_gdf.drop_duplicates(subset='geometry')

#overlap_gdf.to_file(r"C:\Users\I33238\Desktop\Austria\Eubucco_AUT\Test\O1.shp")#, layer="o1", driver="GPKG")
#non_overlap_gdf.to_file(r"C:\Users\I33238\Desktop\Austria\Eubucco_AUT\Test\NO1.shp")#, layer="no1", driver="GPKG")
#intersected_gdf.to_file(r"C:\Users\I33238\Desktop\Italy\estonia-251106-free\Result_LU\Intersected.shp")
#(r"C:\Users\I33238\Desktop\Austria\Eubucco_AUT\Test\In1.shp")
#processed_gdf.to_file(r"C:\Users\I33238\Desktop\Italy\estonia-251106-free\Result_LU\Processed.shp")

#(r"C:\Users\I33238\Desktop\Austria\Eubucco_AUT\Test\P1.shp")
#merged_gdf.to_file(r"C:\Users\I33238\Desktop\Italy\estonia-251106-free.shp\Result_LU\merged.shp")
print("Shapefiles created successfully!")

AttributeError: 'NoneType' object has no attribute 'is_valid'

In [ ]:
#print(gpd.__version__)
non_overlap_gdf.to_file(r"C:\Users\I33238\Desktop\Italy\estonia-251106-free\Result_LU\R1\N2.shp")
processed_gdf.to_file(r"C:\Users\I33238\Desktop\Italy\estonia-251106-free\Result_LU\R1\P2.shp")

In [ ]:
import geopandas as gpd
from shapely.validation import make_valid
import dask_geopandas as dgpd
from dask.distributed import Client

client = Client()
gdf = gpd.read_file(r"C:\Users\I33238\Desktop\Italy\OSM_Italy\Italy_OSM_Building.gdb",layer="Italy_gis_osm_buildings")

gdf['geometry'] = gdf['geometry'].apply(lambda geom: make_valid(geom) if not geom.is_valid else geom)
gdf = gdf.drop_duplicates(subset='geometry')

if 'fid' in gdf.columns:
    gdf = gdf.rename(columns={'fid': 'fid_orig'})

spatial_index = gdf.sindex
dgdf = dgpd.from_geopandas(gdf, npartitions=8) # change number of partitions

def find_overlaps(geometry):
    possible_matches_index = list(spatial_index.intersection(geometry.bounds))
    possible_matches = gdf.iloc[possible_matches_index]
    return possible_matches[possible_matches.geometry.overlaps(geometry)]


for idx, row in gdf.iterrows():
    overlaps = find_overlaps(row.geometry)
    for overlap_idx, overlap_row in overlaps.iterrows():
        if overlap_idx != idx:  # Avoid self-comparison
            intersection = row.geometry.intersection(overlap_row.geometry)
            if not intersection.is_empty:
                
                area_row = row.geometry.area # Compare areas
                area_overlap = overlap_row.geometry.area

                if area_row > area_overlap:
                    # Remove intersection from the larger polygon (row)
                    new_geom = row.geometry.difference(intersection.buffer(0.00000009)) #0.000000009 for 0.01m as this is in 4326 no you need to convert in 3857
                    gdf.at[idx, 'geometry'] = new_geom
                else:
                    # Remove (overlap_row)
                    new_geom = overlap_row.geometry.difference(intersection.buffer(0.00000009))
                    gdf.at[overlap_idx, 'geometry'] = new_geom


gdf['geometry'] = gdf['geometry'].apply(lambda geom: make_valid(geom) if not geom.is_valid else geom)

gdf.to_file(r"C:\Users\I33238\Desktop\Italy\estonia-251106-free\Result_LU\R2\BF2.gpkg",layer='clean',driver='GPKG')

print("created successfully, overlaps")

Cleaned shapefile created successfully with same feature count and overlaps resolved based on area!
